# External training data: DFL / IDSSE and Metrica

The seven DFL / IDSSE matches (Bassek, Rein, Weber & Memmert, 2025, *Scientific Data* 12:195,
doi:10.1038/s41597-025-04505-y; data on figshare, article 28196177, CC BY 4.0) are Bundesliga and 2. Bundesliga games
from 2022/23, tracked at 25 Hz like BAR-ATH. The three Metrica Sports sample games come from
github.com/metrica-sports/sample-data.

The competition defines the label as D where the corrected tracking marks the ball dead, and otherwise the team the
tracking names as owning it (CHALLENGE.md). DFL position files carry both flags on every ball frame, so we read
`BallStatus` 0 as `D`, and otherwise `BallPossession` 1 as `H` and 2 as `A`. The Metrica files have no such flags, so
we derive possession from their events (`metrica_label_arrays`).

DFL tracking rarely loses the ball or a player. The BAR-ATH tracking loses the ball on 41% of the frames, often for long stretches,
and drops players. `DegradationProfile.from_target` measures this on BAR-ATH without using any labels, and `degrade`
damages each external match the same way: missing-ball stretches drawn from BAR-ATH's own gap lengths, player dropouts
and position jitter. The damaged match then goes through the same preprocessing and feature code as BAR-ATH.

`possession/download_external.ipynb` puts both datasets under `data/external/`. The loader searches that folder
recursively for `*matchinformation*<MATCH>*.xml` and `*positions*<MATCH>*.xml`, downloads a missing DFL file from the
mirror kloppy uses, and caches parsed matches in `artifacts/external/`.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent   # the repository root: notebooks run from possession/
PIPELINE_DIR = REPO_ROOT / "possession"
ARTIFACT_DIR = PIPELINE_DIR / "artifacts"
OUTPUT_DIR = PIPELINE_DIR / "outputs"     

def load_notebook_defs(filename: str, namespace: dict) -> None:
    loaded = namespace.setdefault("_LOADED_NOTEBOOKS", set())
    if filename in loaded:
        return
    loaded.add(filename)
    nb = json.loads((PIPELINE_DIR / filename).read_text(encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] == "code" and "defs" in cell.get("metadata", {}).get("tags", []):
            exec(compile("".join(cell["source"]), f"{filename}::cell{i}", "exec"), namespace)

In [ ]:
load_notebook_defs("helper_model.ipynb", globals())
import dataclasses
import urllib.request
import xml.etree.ElementTree as ET

In [ ]:
IDSSE_MATCHES = {   # match id -> (DFL competition id, fixture)
    "J03WMX": ("DFL-COM-000001", "1. FC Köln vs. FC Bayern München"),
    "J03WN1": ("DFL-COM-000001", "VfL Bochum 1848 vs. Bayer 04 Leverkusen"),
    "J03WPY": ("DFL-COM-000002", "Fortuna Düsseldorf vs. 1. FC Nürnberg"),
    "J03WOH": ("DFL-COM-000002", "Fortuna Düsseldorf vs. SSV Jahn Regensburg"),
    "J03WQQ": ("DFL-COM-000002", "Fortuna Düsseldorf vs. FC St. Pauli"),
    "J03WOY": ("DFL-COM-000002", "Fortuna Düsseldorf vs. F.C. Hansa Rostock"),
    "J03WR9": ("DFL-COM-000002", "Fortuna Düsseldorf vs. 1. FC Kaiserslautern"),
}
EXTERNAL_DIR = Path(os.environ.get("MTAH_EXTERNAL_DIR") or (REPO_ROOT / "data" / "external"))
EXTERNAL_CACHE = ARTIFACT_DIR / "external"
IDSSE_MIRROR = "https://huggingface.co/datasets/pysport/idsse-data/resolve/main/{name}"   # the copy kloppy downloads from


def idsse_filenames(match_id: str) -> Dict[str, str]:
    comp = IDSSE_MATCHES[match_id][0]
    return {"meta": f"DFL_02_01_matchinformation_{comp}_DFL-MAT-{match_id}.xml",
            "positions": f"DFL_04_03_positions_raw_observed_{comp}_DFL-MAT-{match_id}.xml"}


def idsse_files(match_id: str, allow_download: bool = True, verbose: bool = True) -> Dict[str, Path]:
    """Find the match's two DFL files anywhere under data/external/ (e.g. unzipped from figshare), else download them."""
    patterns = {"meta": f"*matchinformation*{match_id}*.xml", "positions": f"*positions*{match_id}*.xml"}
    out = {}
    for kind, name in idsse_filenames(match_id).items():
        hits = sorted(EXTERNAL_DIR.rglob(patterns[kind])) if EXTERNAL_DIR.is_dir() else []
        if hits:
            out[kind] = hits[0]
            continue
        if not allow_download:
            raise FileNotFoundError(f"no {patterns[kind]} under {EXTERNAL_DIR}. Download the IDSSE dataset (figshare article "
                                    "28196177) into data/external/, or set ALLOW_DOWNLOAD = True.")
        EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
        dst, tmp = EXTERNAL_DIR / name, EXTERNAL_DIR / (name + ".part")
        if verbose:
            print(f"[external] downloading {name} ...")
        urllib.request.urlretrieve(IDSSE_MIRROR.format(name=name), tmp)
        tmp.replace(dst)
        out[kind] = dst
    return out

In [ ]:
DFL_SECTIONS = {"firstHalf": 1, "secondHalf": 2}


def parse_dfl_meta(path: Path) -> dict:
    """Match information XML -> which player belongs to the home (0) / away (1) team, pitch size, team names."""
    root = ET.parse(str(path)).getroot()
    sides, player_team, names = [], {}, {}
    for t in root.iter("Team"):
        side = 0 if (t.get("Role") or "").lower() == "home" else 1
        sides.append(side)
        names[side] = t.get("TeamName", "")
        for p in t.iter("Player"):
            player_team[p.get("PersonId")] = side
    if sorted(sides) != [0, 1]:
        raise ValueError(f"{path}: expected one home and one guest team")
    env = root.find(".//Environment")
    pitch = (float(env.get("PitchX", 105.0)), float(env.get("PitchY", 68.0))) if env is not None else (105.0, 68.0)
    return {"player_team": player_team, "pitch_m": pitch, "teams": names}


def parse_dfl_positions(path: Path) -> Dict[int, dict]:
    """Stream the (large) DFL position XML.
    -> {half: {"ball": (N, X, Y, BallPossession, BallStatus) | None, "players": {PersonId: (N, X, Y)}}}"""
    out = {h: {"ball": None, "players": {}} for h in DFL_SECTIONS.values()}
    half = team = pid = None
    n, x, y, bp, bs = [], [], [], [], []
    for ev, el in ET.iterparse(str(path), events=("start", "end")):
        if ev == "start":
            if el.tag == "FrameSet":
                half, team, pid = DFL_SECTIONS.get(el.get("GameSection")), el.get("TeamId"), el.get("PersonId")
                n, x, y, bp, bs = [], [], [], [], []
            continue
        if el.tag == "Frame":
            if half is not None:
                n.append(int(el.get("N")))
                x.append(float(el.get("X") or "nan"))
                y.append(float(el.get("Y") or "nan"))
                if team == "BALL":
                    bp.append(int(el.get("BallPossession") or 0))
                    bs.append(int(el.get("BallStatus") or 0))
            el.clear()
        elif el.tag == "FrameSet":
            if half is not None and n:
                arrs = (np.asarray(n, np.int64), np.asarray(x, np.float32), np.asarray(y, np.float32))
                if team == "BALL":
                    out[half]["ball"] = arrs + (np.asarray(bp, np.int8), np.asarray(bs, np.int8))
                else:
                    out[half]["players"][pid] = arrs
            el.clear()
    return out


def dfl_to_tracking(meta: dict, pos: Dict[int, dict], match_id: str) -> Tuple[TrackingData, np.ndarray]:
    """DFL coordinates (metres, origin at the centre spot, y up) -> the target's frame (0..L, 0..W, y down, like Metrica).
    The two halves are stacked one after the other. Labels from the ball's flags:
    BallStatus 0 -> D (2); otherwise BallPossession 1 -> H (0), 2 -> A (1); anything else -> -1 (unknown)."""
    L, W = meta["pitch_m"]
    halves = [h for h in (1, 2) if pos[h]["ball"] is not None]
    blocks, first_n, off = [], {}, 0
    for h in halves:
        nb = pos[h]["ball"][0]
        first_n[h] = int(nb.min())
        ln = int(nb.max() - nb.min() + 1)
        blocks.append((h, off, off + ln - 1))
        off += ln
    T = off
    pids = sorted({p for h in halves for p in pos[h]["players"] if p in meta["player_team"]})
    col = {p: j for j, p in enumerate(pids)}
    px = np.full((T, len(pids)), np.nan, np.float32)
    py = np.full((T, len(pids)), np.nan, np.float32)
    ball = np.full((T, 2), np.nan, np.float32)
    lab = np.full(T, -1, np.int8)
    for h, a, b in blocks:
        N, X, Y, BP, BS = pos[h]["ball"]
        r = N - first_n[h] + a
        ball[r, 0], ball[r, 1] = X + L / 2, W / 2 - Y
        lab[r] = np.where(BS == 0, 2, np.where(BP == 1, 0, np.where(BP == 2, 1, -1)))
        for p, (Np, Xp, Yp) in pos[h]["players"].items():
            if p not in col:
                continue
            r = Np - first_n[h] + a
            ok = (r >= a) & (r <= b)
            px[r[ok], col[p]] = Xp[ok] + L / 2
            py[r[ok], col[p]] = W / 2 - Yp[ok]
    team = np.array([meta["player_team"][p] for p in pids], np.int8)
    td = TrackingData(px=px, py=py, team=team, ball_xy=ball, half_blocks=blocks, fps=25, pitch_m=(float(L), float(W)),
                      known_dead_ranges=[], signature=f"idsse-{match_id}")
    return td, lab


def load_idsse_match(match_id: str, allow_download: bool = True, verbose: bool = True) -> Tuple[TrackingData, np.ndarray]:
    """Clean DFL tracking + per-frame labels for one IDSSE match (parsed once, then cached in artifacts/external/)."""
    EXTERNAL_CACHE.mkdir(parents=True, exist_ok=True)
    tpath, lpath = EXTERNAL_CACHE / f"{match_id}_tracking.npz", EXTERNAL_CACHE / f"{match_id}_labels.npy"
    if tpath.is_file() and lpath.is_file():
        return TrackingData.load(tpath), np.load(lpath)
    files = idsse_files(match_id, allow_download, verbose)
    t0 = time.time()
    meta = parse_dfl_meta(files["meta"])
    td, lab = dfl_to_tracking(meta, parse_dfl_positions(files["positions"]), match_id)
    td.save(tpath)
    np.save(lpath, lab)
    if verbose:
        cnt = {k: int((lab == v).sum()) for k, v in LABEL_TO_ID.items()}
        print(f"[external] {match_id} {IDSSE_MATCHES.get(match_id, ('', ''))[1]}: parsed in {time.time() - t0:.0f}s | "
              f"{td.n_frames} frames, {td.n_slots} players | labels {cnt}, unknown {int((lab < 0).sum())}")
    return td, lab

In [ ]:
# ---------------------------------------------------------------- Metrica sample games (public: github.com/metrica-sports/sample-data)
# Raw tracking + events, the same provider and EPTS layout as BAR-ATH (game 3; games 1 and 2 are the older CSV layout).
# There is no possession flag in these files, so one is derived from the events (see metrica_label_arrays).
METRICA_DIR = EXTERNAL_DIR / "metrica"
METRICA_MATCHES = {   # match id -> (folder, description)
    "METRICA1": ("Sample_Game_1", "Metrica sample game 1 (CSV)"),
    "METRICA2": ("Sample_Game_2", "Metrica sample game 2 (CSV)"),
    "METRICA3": ("Sample_Game_3", "Metrica sample game 3 (EPTS, like BAR-ATH)"),
}
METRICA_PITCH_M = (105.0, 68.0)                 # the samples are normalised 0-1 on a standard pitch

DEAD_UNTIL_CONTACT = True                       # label convention: a restart stays Dead until ANOTHER player touches the ball
DEAD_CONTACT_MAX_S = 6.0
CONTACT_TAG = "contact1"                         # part of the cache file name: bump it if the convention code changes

_METRICA_POSSESS = ("PASS", "SHOT", "CARRY", "RECOVERY", "SET PIECE")
_METRICA_ACTIONS = ("PASS", "SHOT", "CARRY", "RECOVERY", "CHALLENGE", "BALL LOST")


def _metrica_events_csv(path: Path) -> pd.DataFrame:
    e = pd.read_csv(path)
    return pd.DataFrame({
        "team": np.where(e["Team"].str.lower() == "home", 0, 1).astype(np.int8),
        "type": e["Type"].astype(str).str.strip().str.upper(), "subtype": e["Subtype"].fillna("").astype(str).str.strip().str.upper(),
        "start": e["Start Frame"].astype(int), "end": e["End Frame"].astype(int),
        "frm": e["From"].fillna("").astype(str), "to": e["To"].fillna("").astype(str)})


def _metrica_events_json(path: Path, home_team_id: str) -> pd.DataFrame:
    rows = []
    for r in json.loads(Path(path).read_text(encoding="utf-8"))["data"]:
        st = r.get("subtypes")
        st = "-".join(x["name"] for x in st) if isinstance(st, list) else (st or {}).get("name", "") if isinstance(st, dict) else ""
        rows.append({"team": 0 if r["team"]["id"] == home_team_id else 1, "type": str(r["type"]["name"]).strip().upper(), "subtype": st.strip().upper(),
                     "start": int(r["start"]["frame"]), "end": int(r["end"]["frame"]),
                     "frm": (r.get("from") or {}).get("id", ""), "to": (r.get("to") or {}).get("id", "")})
    return pd.DataFrame(rows).astype({"team": np.int8})


def _metrica_tracking_csv(folder: Path, name: str):
    """Games 1 & 2: one CSV per side, 3 header rows, then Period, Frame, Time, (x, y) per player ..., ball x, y."""
    sides = {s: pd.read_csv(folder / f"{name}_RawTrackingData_{s}_Team.csv", header=None, skiprows=3) for s in ("Home", "Away")}
    hd, ad = sides["Home"], sides["Away"]
    assert (hd[1].to_numpy() == ad[1].to_numpy()).all(), "home and away tracking are not frame-aligned"
    nh, na = (hd.shape[1] - 5) // 2, (ad.shape[1] - 5) // 2
    hx, ax = hd.iloc[:, 3:3 + 2 * nh].to_numpy(np.float32), ad.iloc[:, 3:3 + 2 * na].to_numpy(np.float32)
    px, py = np.concatenate([hx[:, 0::2], ax[:, 0::2]], axis=1), np.concatenate([hx[:, 1::2], ax[:, 1::2]], axis=1)
    return (hd[1].to_numpy(np.int64), hd[0].to_numpy(np.int64), px, py, hd.iloc[:, -2:].to_numpy(np.float32),
            np.array([0] * nh + [1] * na, np.int8), None)


def _metrica_tracking_epts(folder: Path, name: str):
    """Game 3: EPTS metadata + 'frame:x,y;x,y;...:ball_x,ball_y' lines (one DataFormatSpecification per substitution stretch)."""
    meta = ET.parse(str(folder / f"{name}_metadata.xml")).getroot()
    team_of = {p.get("id"): p.get("teamId") for p in meta.iter("Player")}
    home_id = meta.find(".//Score").get("idLocalTeam")
    chan_player = {c.get("id"): c.get("playerId") for c in meta.iter("PlayerChannel")}
    lines = (folder / f"{name}_tracking.txt").read_text(encoding="utf-8").replace("\r", "").split("\n")
    specs, cols = [], {}
    for spec in meta.iter("DataFormatSpecification"):
        refs = [r.get("playerChannelId") for r in spec.iter("PlayerChannelRef")]
        pids = [chan_player[r] for r in refs[0::2]]
        for p in pids:
            cols.setdefault(p, len(cols))
        specs.append((int(spec.get("startFrame")), int(spec.get("endFrame")), [cols[p] for p in pids]))
    T = max(e for _, e, _ in specs)
    px, py = np.full((T, len(cols)), np.nan, np.float32), np.full((T, len(cols)), np.nan, np.float32)
    ball = np.full((T, 2), np.nan, np.float32)
    for a, b, idx in specs:
        k = len(idx)
        arr = np.fromstring(",".join(lines[a - 1:b]).replace(":", ",").replace(";", ","), sep=",", dtype=np.float64).reshape(-1, 1 + 2 * k + 2)
        rows = arr[:, 0].astype(np.int64) - 1
        px[rows[:, None], np.asarray(idx)[None, :]] = arr[:, 1:1 + 2 * k:2]
        py[rows[:, None], np.asarray(idx)[None, :]] = arr[:, 2:2 + 2 * k:2]
        ball[rows] = arr[:, -2:]
    prm = {p.findtext("Name"): p.findtext("Value") for p in meta.iter("ProviderParameter")}
    frame = np.arange(1, T + 1, dtype=np.int64)
    period = np.where(frame <= int(prm["first_half_end"]), 1, 2).astype(np.int64)
    team = np.array([0 if team_of[p] == home_id else 1 for p in sorted(cols, key=cols.get)], np.int8)
    return frame, period, px, py, ball, team, home_id


def metrica_dead_mask(ball_nan: np.ndarray, ev: pd.DataFrame, fps: int = 25, long_s: float = 3.0, short_s: float = 0.4) -> np.ndarray:
    """True = ball dead. The tracking drops the ball while it is out of play, so a long missing-ball stretch is a stoppage;
    a shorter one only when the events confirm it (a ball out / foul just before, or a restart just after)."""
    dead = np.zeros(len(ball_nan), bool)
    stop_f = ev.loc[ev.type.isin(["BALL OUT", "FAULT RECEIVED", "CARD"]), "start"].to_numpy()
    restart_f = ev.loc[(ev.type == "SET PIECE") | ev.subtype.str.contains("GOAL KICK"), "start"].to_numpy()
    for a, b in BallInfiller._gaps(ball_nan):
        n = b - a
        if n >= long_s * fps:
            dead[a:b] = True
        elif n >= short_s * fps:
            fa, fb = a + 1, b + 1
            if ((stop_f >= fa - 2 * fps) & (stop_f <= fa + fps)).any() or ((restart_f >= fb - fps) & (restart_f <= fb + fps)).any():
                dead[a:b] = True
    return dead


def metrica_owner(ev: pd.DataFrame, T: int, half_blocks) -> np.ndarray:
    """(T,) int8: the team that last acted on the ball (0 home, 1 away, -1 before the first action of a half)."""
    owner = np.full(T, -1, np.int8)
    e = ev[ev.type.isin(_METRICA_POSSESS)].sort_values("start", kind="stable")
    owner[np.clip(e.start.to_numpy() - 1, 0, T - 1)] = e.team.to_numpy(np.int8)
    for _, a, b in half_blocks:
        seg = owner[a:b + 1]
        last = np.maximum.accumulate(np.where(seg >= 0, np.arange(len(seg)), -1))
        seg[:] = np.where(last >= 0, seg[np.maximum(last, 0)], -1)
    return owner


def metrica_contact_ends(ev: pd.DataFrame, dead: np.ndarray, fps: int = 25, max_s: float = DEAD_CONTACT_MAX_S) -> List[Tuple[int, int]]:
    """(restart-kick row, first-touch-by-another-player row) per stoppage, for the 'D until contact' convention."""
    e = ev.sort_values("start", kind="stable").reset_index(drop=True)
    starts = e.start.to_numpy()
    pairs = []
    for a, b in BallInfiller._gaps(dead):
        if b >= len(dead):
            continue
        fb = b + 1
        i0 = int(np.searchsorted(starts, fb - 3))
        taker = e.iloc[i0:i0 + 8]
        taker = taker[(taker.start <= fb + 4) & taker.type.isin(["SET PIECE", "PASS", "SHOT", "CARRY"]) & (taker.frm != "")]
        if taker.empty:
            continue
        first = taker.frm.iloc[0]
        flight = taker.end[taker.type.isin(["PASS", "SHOT"]) & (taker.end > fb + 2)]
        rest = e.iloc[i0:]
        other = rest[(rest.start > fb + 2) & (rest.frm != first) & (rest.frm != "") & rest.type.isin(_METRICA_ACTIONS)].head(1)
        cands = ([int(flight.min())] if len(flight) else []) + ([int(other.start.iloc[0])] if len(other) else [])
        if cands and b + 2 < min(cands) - 1 <= b + int(max_s * fps):
            pairs.append((b, min(cands) - 1))
    return pairs


def metrica_label_arrays(ev: pd.DataFrame, ball: np.ndarray, half_blocks, fps: int = 25):
    """-> (labels 'D until the restart kick', labels 'D until another player touches it', dead mask). int8: 0 H, 1 A, 2 D, -1 unknown."""
    T = len(ball)
    dead = metrica_dead_mask(~np.isfinite(ball).all(axis=1), ev, fps)
    lab = np.where(dead, 2, metrica_owner(ev, T, half_blocks)).astype(np.int8)
    lab_contact = lab.copy()
    for k, c in metrica_contact_ends(ev, dead, fps):
        lab_contact[k:c] = 2
    return lab, lab_contact, dead


def metrica_fill_ball(ball: np.ndarray, dead: np.ndarray) -> np.ndarray:
    """Freeze the ball at its last known position during a stoppage (what DFL does); short glitches are interpolated."""
    out = ball.copy()
    T = len(out)
    for a, b in BallInfiller._gaps(~np.isfinite(out).all(axis=1)):
        if a == 0 and b == T:
            continue
        if a == 0:
            out[a:b] = out[b]
        elif b == T or dead[a]:
            out[a:b] = out[a - 1]
        else:
            t = (np.arange(a, b) - (a - 1)) / (b - (a - 1))
            out[a:b] = out[a - 1] + t[:, None] * (out[b] - out[a - 1])
    return out


def load_metrica_match(match_id: str, verbose: bool = True) -> Tuple[TrackingData, np.ndarray]:
    """Clean Metrica match + 'D until the restart kick' labels (parsed once, cached in artifacts/external/).
    The 'D until another player touches it' labels are cached too (<id>_labels_<CONTACT_TAG>.npy)."""
    EXTERNAL_CACHE.mkdir(parents=True, exist_ok=True)
    tpath, lpath = EXTERNAL_CACHE / f"{match_id}_tracking.npz", EXTERNAL_CACHE / f"{match_id}_labels.npy"
    cpath = EXTERNAL_CACHE / f"{match_id}_labels_{CONTACT_TAG}.npy"
    if tpath.is_file() and lpath.is_file() and cpath.is_file():
        return TrackingData.load(tpath), np.load(lpath)
    name, desc = METRICA_MATCHES[match_id]
    folder = METRICA_DIR / name
    t0 = time.time()
    if (folder / f"{name}_tracking.txt").is_file():
        frame, period, px, py, ball, team, home_id = _metrica_tracking_epts(folder, name)
        ev = _metrica_events_json(folder / f"{name}_events.json", home_id)
    elif (folder / f"{name}_RawTrackingData_Home_Team.csv").is_file():
        frame, period, px, py, ball, team, _ = _metrica_tracking_csv(folder, name)
        ev = _metrica_events_csv(folder / f"{name}_RawEventsData.csv")
    else:
        raise FileNotFoundError(f"no Metrica sample data in {folder}: put the sample games under data/external/metrica/")
    assert frame[0] == 1 and (np.diff(frame) == 1).all(), "frames are expected to run 1..T without gaps"
    L, W = METRICA_PITCH_M
    scale = np.array([L, W], np.float32)
    px, py, ball = px * L, py * W, ball * scale
    blocks = [(int(h), int(np.flatnonzero(period == h).min()), int(np.flatnonzero(period == h).max())) for h in (1, 2) if (period == h).any()]
    lab, lab_contact, dead = metrica_label_arrays(ev, ball, blocks)
    td = TrackingData(px=px, py=py, team=team, ball_xy=metrica_fill_ball(ball, dead), half_blocks=blocks, fps=25, pitch_m=(L, W),
                      known_dead_ranges=[], signature=f"metrica-{match_id}")
    td.save(tpath)
    np.save(lpath, lab)
    np.save(cpath, lab_contact)
    if verbose:
        cnt = {k: int((lab == v).sum()) for k, v in LABEL_TO_ID.items()}
        print(f"[external] {match_id} {desc}: parsed in {time.time() - t0:.0f}s | {td.n_frames} frames, {td.n_slots} players | labels {cnt}, unknown {int((lab < 0).sum())}")
    return td, lab


def extend_dead_until_contact(td: TrackingData, lab: np.ndarray, max_s: float = DEAD_CONTACT_MAX_S, touch_m: float = 1.8, dv_min: float = 2.5,
                              min_dead_s: float = 1.0, skip_frames: int = 12, lag: int = 3, verbose: bool = False) -> np.ndarray:
    """Target label convention: after a restart the label stays D until ANOTHER player makes contact. Used for DFL matches
    (the DFL flag turns the ball alive at the kick); every dead -> live change is moved to the first contact of a second player."""
    fps = td.fps
    out = lab.copy()
    ball = td.ball_xy.copy()
    ball[lab == 2] = np.nan
    bx, by = nan_gaussian(ball[:, 0], 2.0), nan_gaussian(ball[:, 1], 2.0)
    with np.errstate(invalid="ignore"):
        vx, vy = np.gradient(bx) * fps, np.gradient(by) * fps
        dv = np.hypot(_shift(vx, -lag) - _shift(vx, lag), _shift(vy, -lag) - _shift(vy, lag))
    n_win = int(max_s * fps) + 1
    for _, a, b in td.half_blocks:
        for s, e in BallInfiller._gaps(lab[a:b + 1] == 2):
            if e - s < min_dead_s * fps or a + e > b:
                continue
            k = a + e
            w1 = min(k + n_win, b + 1)
            dist = np.hypot(td.px[k:w1] - td.ball_xy[k:w1, 0:1], td.py[k:w1] - td.ball_xy[k:w1, 1:2])
            dist = np.where(np.isfinite(dist), dist, np.inf)
            if not np.isfinite(dist[:4]).any():
                continue
            dist[:, int(np.argmin(dist[:4].min(axis=0)))] = np.inf
            with np.errstate(invalid="ignore"):
                hit = np.flatnonzero((dist.min(axis=1) <= touch_m) & (dv[k:w1] >= dv_min) & (np.arange(w1 - k) >= skip_frames))
            dead_again = np.flatnonzero(lab[k:w1] == 2)
            c = k + int(hit[0]) if len(hit) else None
            if dead_again.size and (c is None or k + int(dead_again[0]) < c):
                c = k + int(dead_again[0])
            if c is not None and c > k:
                out[k:c] = 2
    return out


def load_external_match(match_id: str, allow_download: bool = True, verbose: bool = True) -> Tuple[TrackingData, np.ndarray]:
    """Any external match (DFL/IDSSE or Metrica): clean tracking + labels with the active dead-ball convention."""
    if match_id in METRICA_MATCHES:
        td, lab = load_metrica_match(match_id, verbose)
        if not DEAD_UNTIL_CONTACT:
            return td, lab
        cpath = EXTERNAL_CACHE / f"{match_id}_labels_{CONTACT_TAG}.npy"
        return td, (np.load(cpath) if cpath.is_file() else lab)
    td, lab = load_idsse_match(match_id, allow_download, verbose)
    if not DEAD_UNTIL_CONTACT:
        return td, lab
    cpath = EXTERNAL_CACHE / f"{match_id}_labels_{CONTACT_TAG}.npy"
    if not cpath.is_file():
        np.save(cpath, extend_dead_until_contact(td, lab, verbose=verbose))
    return td, np.load(cpath)


def is_native_match(match_id: str) -> bool:
    """The 3 Metrica sample games share BAR-ATH's native EPTS provider -> the target-domain proxy."""
    return match_id in METRICA_MATCHES


In [ ]:
def _run_lengths(mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """(lengths of the True runs, lengths of the False runs)."""
    if len(mask) == 0:
        return np.array([], int), np.array([], int)
    edges = np.concatenate([[0], np.flatnonzero(np.diff(mask.astype(np.int8))) + 1, [len(mask)]])
    lens, vals = np.diff(edges), mask[edges[:-1]]
    return lens[vals], lens[~vals]


@dataclass
class DegradationProfile:
    """How the target's tracking is broken - measured on the target itself, no labels involved."""
    ball_gap_runs: np.ndarray            # lengths (frames) of the target's missing-ball stretches
    ball_obs_runs: np.ndarray            # lengths (frames) of its observed-ball stretches
    player_miss_frac: float              # share of on-pitch players missing in an average frame
    player_gap_mean: float = 25.0        # mean length (frames) of one player dropout
    pos_noise_m: float = 0.10            # extra jitter on every position (video tracking is noisier than DFL's)

    @classmethod
    def from_target(cls, raw: TrackingData, players_on_pitch: int = 22, **kw) -> "DegradationProfile":
        gaps, obs, valid = [], [], []
        for _, a, b in raw.half_blocks:
            g, o = _run_lengths(~np.isfinite(raw.ball_xy[a:b + 1]).all(axis=1))
            gaps.append(g)
            obs.append(o)
            valid.append(np.isfinite(raw.px[a:b + 1]).sum(axis=1))
        miss = float(np.clip(1 - np.concatenate(valid).mean() / players_on_pitch, 0.0, 0.9))
        return cls(ball_gap_runs=np.concatenate(gaps), ball_obs_runs=np.concatenate(obs), player_miss_frac=miss, **kw)

    def summary(self, fps: int = 25) -> Dict[str, float]:
        g, o = self.ball_gap_runs, self.ball_obs_runs
        return {"ball_missing_pct": round(100 * g.sum() / max(g.sum() + o.sum(), 1), 1), "ball_gaps": int(len(g)),
                "ball_gap_median_s": round(float(np.median(g)) / fps, 1) if len(g) else 0.0,
                "ball_gap_mean_s": round(float(np.mean(g)) / fps, 1) if len(g) else 0.0,
                "ball_gap_p90_s": round(float(np.quantile(g, 0.9)) / fps, 1) if len(g) else 0.0,
                "player_missing_pct": round(100 * self.player_miss_frac, 1), "pos_noise_m": self.pos_noise_m}


def _alternating_mask(n: int, on_runs: np.ndarray, off_runs: np.ndarray, rng) -> np.ndarray:
    """True = missing. Alternates observed / missing stretches whose lengths are resampled from the target's own."""
    mask = np.zeros(n, bool)
    if len(off_runs) == 0:
        return mask
    if len(on_runs) == 0:
        mask[:] = True
        return mask
    missing = rng.random() < off_runs.sum() / (off_runs.sum() + on_runs.sum())
    t, first = 0, True
    while t < n:
        L = int(rng.choice(off_runs if missing else on_runs))
        if first:                                         # start at a random point inside the first stretch
            L, first = int(rng.integers(1, L + 1)), False
        if missing:
            mask[t:t + L] = True
        t += L
        missing = not missing
    return mask


def _dropout_mask(n: int, miss_frac: float, gap_mean: float, rng) -> np.ndarray:
    """True = missing; exponential present / missing stretches with the requested missing share."""
    mask = np.zeros(n, bool)
    if miss_frac <= 0:
        return mask
    on_mean = gap_mean * (1 - miss_frac) / miss_frac
    t, missing = int(rng.exponential(on_mean)), False
    while t < n:
        L = max(1, int(rng.exponential(gap_mean if not missing else on_mean)))
        if not missing:
            mask[t:t + L] = True
        t += L
        missing = not missing
    return mask


def degrade(td: TrackingData, prof: DegradationProfile, seed: int = 0) -> TrackingData:
    """Break a clean match the way the target is broken (missing ball stretches, player dropouts, jitter)."""
    rng = np.random.default_rng(seed)
    px, py, ball = td.px.copy(), td.py.copy(), td.ball_xy.copy()
    for _, a, b in td.half_blocks:
        n = b - a + 1
        ball[a:b + 1][_alternating_mask(n, prof.ball_obs_runs, prof.ball_gap_runs, rng)] = np.nan
        for j in range(px.shape[1]):
            m = _dropout_mask(n, prof.player_miss_frac, prof.player_gap_mean, rng)
            px[a:b + 1, j][m] = np.nan
            py[a:b + 1, j][m] = np.nan
    if prof.pos_noise_m > 0:
        for arr in (px, py, ball):
            arr += rng.normal(0.0, prof.pos_noise_m, arr.shape).astype(np.float32)   # NaN stays NaN
    return TrackingData(px=px, py=py, team=td.team, ball_xy=ball, half_blocks=list(td.half_blocks), fps=td.fps,
                        pitch_m=td.pitch_m, known_dead_ranges=list(td.known_dead_ranges), signature=f"{td.signature}-deg{seed}")

In [ ]:
def external_features(match_id: str, prof: DegradationProfile, seed: int, pre_cfg: PreprocessConfig, feat_cfg: FeatureConfig,
                      allow_download: bool = True, verbose: bool = True) -> Tuple[pd.DataFrame, np.ndarray, list]:
    """Clean external match (DFL or Metrica) -> degraded like the target -> the same preprocessing and features as the target.
    -> (model features (T, F) float32, labels (T,) int8, half_blocks)"""
    raw, lab = load_external_match(match_id, allow_download, verbose)
    clean_ = TrackingPreprocessor(dataclasses.replace(pre_cfg, pitch_m=tuple(raw.pitch_m))).run(degrade(raw, prof, seed), verbose=False)
    F_ = SpatialFeatureExtractor(feat_cfg).tabular(clean_)
    return WindowFeatureBuilder().transform(F_, clean_.half_blocks), lab, clean_.half_blocks


@dataclass
class ExternalSplit:
    X: pd.DataFrame
    y: np.ndarray
    blocks: list


@dataclass
class ExternalSets:
    X_train: pd.DataFrame                           # sub-sampled rows of the training matches (all augmented copies)
    y_train: np.ndarray
    val: Dict[str, ExternalSplit]                   # validation matches, EVERY frame (the decoder needs whole sequences)
    labels: Dict[str, Tuple[np.ndarray, list]]      # clean labels + half blocks of every match

    def val_rows(self, stride: int = 5) -> Tuple[pd.DataFrame, np.ndarray]:
        Xs, ys = [], []
        for v in self.val.values():
            idx = np.flatnonzero(v.y >= 0)[::stride]
            Xs.append(v.X.iloc[idx])
            ys.append(v.y[idx])
        return pd.concat(Xs, ignore_index=True), np.concatenate(ys)

    def all_rows(self, stride: int = 5) -> Tuple[pd.DataFrame, np.ndarray]:
        Xv, yv = self.val_rows(stride)
        return pd.concat([self.X_train, Xv], ignore_index=True), np.concatenate([self.y_train, yv])


def build_external_sets(train_ids, val_ids, prof: DegradationProfile, pre_cfg: PreprocessConfig, feat_cfg: FeatureConfig,
                        n_aug: int = 2, stride: int = 5, allow_download: bool = True, seed: int = 0,
                        verbose: bool = True) -> ExternalSets:
    """Training matches: ``n_aug`` independently degraded copies each, every ``stride``-th frame (a different offset per copy).
    Validation matches: one degraded copy, all frames."""
    Xs, ys, labels = [], [], {}
    for k, mid in enumerate(train_ids):
        for aug in range(n_aug):
            t0 = time.time()
            Fw_, lab, blocks = external_features(mid, prof, seed + 100 * k + aug, pre_cfg, feat_cfg, allow_download, verbose)
            keep = np.zeros(len(lab), bool)
            keep[(aug * stride) // max(n_aug, 1)::stride] = True
            idx = np.flatnonzero(keep & (lab >= 0))
            Xs.append(Fw_.iloc[idx].reset_index(drop=True))
            ys.append(lab[idx])
            labels[mid] = (lab, blocks)
            if verbose:
                print(f"[external] train {mid} copy {aug + 1}/{n_aug}: {len(idx):,} rows ({time.time() - t0:.0f}s)")
    val = {}
    for k, mid in enumerate(val_ids):
        t0 = time.time()
        Fw_, lab, blocks = external_features(mid, prof, seed + 10_000 + k, pre_cfg, feat_cfg, allow_download, verbose)
        val[mid] = ExternalSplit(Fw_, lab, blocks)
        labels[mid] = (lab, blocks)
        if verbose:
            print(f"[external] val   {mid}: {len(lab):,} frames ({time.time() - t0:.0f}s)")
    return ExternalSets(pd.concat(Xs, ignore_index=True), np.concatenate(ys), val, labels)